In [0]:
import json
import random
import time
import uuid
from datetime import datetime, timezone
from typing import Any, Dict, List
import urllib.error
import urllib.request

from azure.eventhub import EventData, EventHubProducerClient
from azure.eventhub.exceptions import EventHubError

# ---------------------------------------------------------------------------
# 1. Configuration & Secret Retrieval (never log or print secret values)
# ---------------------------------------------------------------------------
SECRET_SCOPE = "kv-aircargo-scope"
SEND_CONN_SECRET_KEY = "eh-flight-telemetry-send-conn"
EVENT_HUB_NAME = "flight-telemetry"
OPENSKY_STATES_URL = "https://opensky-network.org/api/states/all"
MAX_FLIGHTS = 25
REQUEST_TIMEOUT_SECONDS = 10

conn_str = dbutils.secrets.get(scope=SECRET_SCOPE, key=SEND_CONN_SECRET_KEY)
assert conn_str and len(conn_str.strip()) > 0, "Empty Event Hubs connection string"


# ---------------------------------------------------------------------------
# 2. Fetch live telemetry from OpenSky API with deterministic fallback
# ---------------------------------------------------------------------------
def build_synthetic_cargo_events(count: int = MAX_FLIGHTS) -> List[Dict[str, Any]]:
    """Generate realistic air cargo telemetry events when external API is unavailable."""
    corridors = [
        ("WAW", "FRA", "LH8301", 52.1657, 20.9671, 50.0379, 8.5622),
        ("FRA", "ORD", "LH8210", 50.0379, 8.5622, 41.9742, -87.9073),
        ("AMS", "HKG", "KL0887", 52.3105, 4.7683, 22.3080, 113.9185),
        ("LEJ", "CVG", "QY7912", 51.4324, 12.2416, 39.0488, -84.6678),
        ("WAW", "LHR", "LO0281", 52.1657, 20.9671, 51.4700, -0.4543),
    ]
    now_iso = datetime.now(timezone.utc).isoformat()
    events: List[Dict[str, Any]] = []

    for idx in range(count):
        origin, dest, flight_code, lat0, lon0, lat1, lon1 = corridors[idx % len(corridors)]
        progress = round(random.uniform(0.05, 0.95), 3)
        lat = round(lat0 + (lat1 - lat0) * progress + random.uniform(-0.4, 0.4), 5)
        lon = round(lon0 + (lon1 - lon0) * progress + random.uniform(-0.4, 0.4), 5)
        alt_m = round(random.uniform(7500.0, 11800.0), 1)
        vel_mps = round(random.uniform(195.0, 255.0), 1)

        events.append({
            "event_id": str(uuid.uuid4()),
            "source": "synthetic_cargo_fallback",
            "event_timestamp_utc": now_iso,
            "icao24": f"48{idx:04x}"[:6],
            "callsign": f"{flight_code}{idx:02d}",
            "origin_airport": origin,
            "destination_airport": dest,
            "cargo_uld_count": random.randint(4, 28),
            "cargo_weight_kg": random.randint(3500, 78000),
            "latitude": lat,
            "longitude": lon,
            "baro_altitude_m": alt_m,
            "velocity_mps": vel_mps,
            "on_ground": False,
        })

    return events


def fetch_opensky_events(max_flights: int = MAX_FLIGHTS) -> List[Dict[str, Any]]:
    """Fetch current aircraft state vectors from OpenSky API and normalize payload."""
    req = urllib.request.Request(
        OPENSKY_STATES_URL,
        headers={"User-Agent": "AirCargoLakehouse/1.0 (portfolio-dev)"},
    )
    with urllib.request.urlopen(req, timeout=REQUEST_TIMEOUT_SECONDS) as resp:
        if resp.status != 200:
            raise RuntimeError(f"Unexpected OpenSky HTTP status: {resp.status}")
        raw_body = resp.read().decode("utf-8")

    payload = json.loads(raw_body)
    states = payload.get("states") or []
    now_iso = datetime.now(timezone.utc).isoformat()
    events: List[Dict[str, Any]] = []

    for row in states:
        # OpenSky state vector indices: 0=icao24, 1=callsign, 2=origin_country,
        # 5=longitude, 6=latitude, 7=baro_altitude, 8=on_ground, 9=velocity
        if len(row) < 10:
            continue
        lon, lat = row[5], row[6]
        if lon is None or lat is None:
            continue

        callsign = (row[1] or "").strip() or "UNKNOWN"
        events.append({
            "event_id": str(uuid.uuid4()),
            "source": "opensky_api",
            "event_timestamp_utc": now_iso,
            "icao24": row[0],
            "callsign": callsign,
            "origin_country": row[2],
            "cargo_uld_count": random.randint(2, 24),
            "cargo_weight_kg": random.randint(2000, 65000),
            "latitude": float(lat),
            "longitude": float(lon),
            "baro_altitude_m": float(row[7]) if row[7] is not None else None,
            "velocity_mps": float(row[9]) if row[9] is not None else None,
            "on_ground": bool(row[8]),
        })

        if len(events) >= max_flights:
            break

    if not events:
        raise RuntimeError("OpenSky returned zero usable flight state records")

    return events


try:
    telemetry_events = fetch_opensky_events(MAX_FLIGHTS)
    print(f"Fetched {len(telemetry_events)} live events from OpenSky API.")
except (urllib.error.URLError, TimeoutError, json.JSONDecodeError, RuntimeError) as exc:
    print(f"OpenSky unavailable ({type(exc).__name__}: {exc}). Switching to synthetic fallback.")
    telemetry_events = build_synthetic_cargo_events(MAX_FLIGHTS)
    print(f"Generated {len(telemetry_events)} synthetic cargo telemetry events.")


# ---------------------------------------------------------------------------
# 3. Publish events in batches to Azure Event Hubs
# ---------------------------------------------------------------------------
def publish_to_event_hub(
    connection_str: str,
    event_hub_name: str,
    records: List[Dict[str, Any]],
) -> int:
    """Send JSON records using EventDataBatch so batch size limits are respected."""
    sent_count = 0
    producer = EventHubProducerClient.from_connection_string(
        conn_str=connection_str,
        eventhub_name=event_hub_name,
    )

    try:
        batch = producer.create_batch()
        for record in records:
            event_data = EventData(json.dumps(record))
            try:
                batch.add(event_data)
                sent_count += 1
            except ValueError:
                # Current batch reached Event Hubs max size; flush and start a new batch
                producer.send_batch(batch)
                batch = producer.create_batch()
                batch.add(event_data)
                sent_count += 1

        if len(batch) > 0:
            producer.send_batch(batch)
    except EventHubError as exc:
        raise RuntimeError(f"Failed to publish events to '{event_hub_name}': {exc}") from exc
    finally:
        producer.close()

    return sent_count


published = publish_to_event_hub(conn_str, EVENT_HUB_NAME, telemetry_events)
print(f"Successfully published {published} events to Event Hub '{EVENT_HUB_NAME}'.")